# NBA Team Style Features
Computes 18 team-level style features from the 2025-26 season SQLite database and saves them to `data/team_features.csv`.

Features are grouped into five categories:
- **Offensive** (9 features): shooting profile, efficiency, and ball-movement tendencies
- **Defensive** (5 features): opponent shooting profile and turnover generation
- **Pace** (1 feature): average possessions per 48 minutes
- **Player Composition** (3 features): RAPM-weighted quality, roster concentration, rotation depth

Output: `data/team_features.csv` (30 rows × 20 columns including team_id and abbrev)

In [ ]:
import sqlite3
import os
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

# Add project root to sys.path so we can import config
sys.path.insert(0, os.path.abspath('..'))
from config import DB_PATH, NBA_TEAM_MAP

print("DB path:", DB_PATH)
print("DB exists:", os.path.exists(DB_PATH))

In [ ]:
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

# Verify the DB is populated
n_games = conn.execute("SELECT COUNT(*) FROM games WHERE season = '2025-26'").fetchone()[0]
n_poss = conn.execute("SELECT COUNT(*) FROM possessions WHERE season = '2025-26'").fetchone()[0]
n_ratings = conn.execute("SELECT COUNT(*) FROM current_ratings").fetchone()[0]

print(f"Games (2025-26):       {n_games:,}")
print(f"Possessions (2025-26): {n_poss:,}")
print(f"Current ratings:       {n_ratings:,}")

## Offensive Features

Computed from possessions where `offense_team_id = team` for the full 2025-26 season.

Note on `transition_rate`: pbpstats encodes possession start types using shot-context labels
(e.g. `OffLiveBallTurnover`, `OffAtRimMake`). There is no literal `'Transition'` value.
The closest proxy for transition offense is `OffLiveBallTurnover` — possessions that start
immediately after the opponent turns the ball over in live play (classic fast-break opportunities).

In [ ]:
sql_off = """
SELECT
    offense_team_id                                          AS team_id,
    CAST(SUM(fg3a) AS REAL)
        / NULLIF(SUM(fg2a) + SUM(fg3a), 0)                  AS three_pt_attempt_rate,
    CAST(SUM(fg3m) AS REAL)
        / NULLIF(SUM(fg3a), 0)                               AS three_pt_pct,
    CAST(SUM(fg2m) AS REAL)
        / NULLIF(SUM(fg2a), 0)                               AS two_pt_pct,
    CAST(SUM(free_throw_points) AS REAL)
        / NULLIF(SUM(points_scored), 0)                      AS ft_point_rate,
    CAST(SUM(fg2a) + SUM(fg3a) AS REAL)
        / COUNT(*)                                           AS fga_per_possession,
    CAST(SUM(turnovers) AS REAL)
        / COUNT(*)                                           AS turnover_rate,
    CAST(SUM(offensive_rebounds) AS REAL)
        / COUNT(*)                                           AS oreb_rate,
    CAST(
        COUNT(CASE WHEN start_type = 'OffLiveBallTurnover' THEN 1 END)
        AS REAL
    ) / CAST(COUNT(*) AS REAL)                               AS transition_rate,
    SUM(points_scored) * 100.0 / COUNT(*)                    AS off_rating
FROM possessions
WHERE season = '2025-26'
GROUP BY offense_team_id
ORDER BY offense_team_id
"""

df_off = pd.read_sql_query(sql_off, conn)
print("Offensive features shape:", df_off.shape)
df_off.head(5)

## Defensive Features

Computed from possessions where `defense_team_id = team` for the full 2025-26 season.
These measure what opponents do *against* this team.

In [ ]:
sql_def = """
SELECT
    defense_team_id                                          AS team_id,
    CAST(SUM(fg3a) AS REAL)
        / NULLIF(SUM(fg2a) + SUM(fg3a), 0)                  AS opp_three_pt_attempt_rate,
    CAST(SUM(fg2m) AS REAL)
        / NULLIF(SUM(fg2a), 0)                               AS opp_two_pt_pct,
    CAST(SUM(turnovers) AS REAL)
        / COUNT(*)                                           AS opp_turnover_rate,
    CAST(SUM(offensive_rebounds) AS REAL)
        / COUNT(*)                                           AS opp_oreb_rate,
    SUM(points_scored) * 100.0 / COUNT(*)                    AS def_rating
FROM possessions
WHERE season = '2025-26'
GROUP BY defense_team_id
ORDER BY defense_team_id
"""

df_def = pd.read_sql_query(sql_def, conn)
print("Defensive features shape:", df_def.shape)
df_def.head(5)

## Pace

`avg_pace` = average `game_pace` (possessions per 48 minutes) across all games
in which this team participated, whether home or away.

In [ ]:
sql_pace = """
SELECT team_id, AVG(game_pace) AS avg_pace
FROM (
    SELECT home_team_id AS team_id, game_pace
    FROM games
    WHERE season = '2025-26'
      AND game_pace IS NOT NULL
    UNION ALL
    SELECT away_team_id AS team_id, game_pace
    FROM games
    WHERE season = '2025-26'
      AND game_pace IS NOT NULL
)
GROUP BY team_id
ORDER BY team_id
"""

df_pace = pd.read_sql_query(sql_pace, conn)
print("Pace features shape:", df_pace.shape)
df_pace.head(5)

## Player Composition Features

Three features derived from full-season possession counts per player per team,
joined with `current_ratings`:

- **`weighted_off_rapm`** — possession-share-weighted average offensive RAPM for the rotation
- **`star_concentration`** — Gini coefficient of normalized possession shares (0 = equal distribution, 1 = one player dominates)
- **`rotation_depth`** — count of players whose share exceeds 0.06 (approximately 15 min/game equivalent)

In [ ]:
sql_player_poss = """
WITH all_players AS (
    SELECT offense_team_id AS team_id, off_player_1 AS player_id
    FROM possessions WHERE season = '2025-26'
    UNION ALL
    SELECT offense_team_id, off_player_2
    FROM possessions WHERE season = '2025-26'
    UNION ALL
    SELECT offense_team_id, off_player_3
    FROM possessions WHERE season = '2025-26'
    UNION ALL
    SELECT offense_team_id, off_player_4
    FROM possessions WHERE season = '2025-26'
    UNION ALL
    SELECT offense_team_id, off_player_5
    FROM possessions WHERE season = '2025-26'
)
SELECT team_id, player_id, COUNT(*) AS poss_count
FROM all_players
GROUP BY team_id, player_id
ORDER BY team_id, poss_count DESC
"""

df_player_poss = pd.read_sql_query(sql_player_poss, conn)
print(f"Player-team possession rows: {len(df_player_poss):,}")

# Load current_ratings for RAPM join
df_ratings = pd.read_sql_query(
    "SELECT player_id, offense FROM current_ratings",
    conn
)
print(f"Players with current ratings: {len(df_ratings):,}")

In [ ]:
def gini_coefficient(values):
    """Compute the Gini coefficient for an array of non-negative values."""
    values = np.array(sorted(values), dtype=float)
    n = len(values)
    if n == 0 or values.sum() == 0:
        return 0.0
    index = np.arange(1, n + 1)
    return float((2 * (index * values).sum()) / (n * values.sum()) - (n + 1) / n)


# Step 1: Filter to players with >= 100 possessions (remove DNP/garbage-time players)
df_filtered = df_player_poss[df_player_poss["poss_count"] >= 100].copy()

# Step 2: Compute normalized share per team
team_totals = df_filtered.groupby("team_id")["poss_count"].sum().rename("team_total")
df_filtered = df_filtered.join(team_totals, on="team_id")
df_filtered["share"] = df_filtered["poss_count"] / df_filtered["team_total"]

# Step 3: Join with current_ratings
df_filtered = df_filtered.merge(df_ratings, on="player_id", how="left")

# Step 4: Compute per-team composition features
comp_rows = []
for team_id, grp in df_filtered.groupby("team_id"):
    shares = grp["share"].values
    off_ratings = grp["offense"].values

    # weighted_off_rapm: only players with a rating contribute
    mask_rated = ~np.isnan(off_ratings)
    if mask_rated.sum() > 0:
        rated_shares = shares[mask_rated]
        rated_offs = off_ratings[mask_rated]
        # Re-normalize shares over rated players only (so weights sum to 1)
        rated_share_sum = rated_shares.sum()
        if rated_share_sum > 0:
            weighted_off_rapm = float((rated_shares / rated_share_sum * rated_offs).sum())
        else:
            weighted_off_rapm = float(np.nan)
    else:
        weighted_off_rapm = float(np.nan)

    star_concentration = gini_coefficient(shares)
    rotation_depth = int((shares > 0.06).sum())

    comp_rows.append({
        "team_id": team_id,
        "weighted_off_rapm": weighted_off_rapm,
        "star_concentration": star_concentration,
        "rotation_depth": rotation_depth,
    })

df_comp = pd.DataFrame(comp_rows).sort_values("team_id").reset_index(drop=True)
print("Composition features shape:", df_comp.shape)
df_comp.head(5)

## Merge All Features

In [ ]:
# Merge on team_id: offensive, defensive, pace, composition
df_features = (
    df_off
    .merge(df_def,  on="team_id", how="outer")
    .merge(df_pace, on="team_id", how="outer")
    .merge(df_comp, on="team_id", how="outer")
)

# Add team abbreviation from NBA_TEAM_MAP
df_features["abbrev"] = df_features["team_id"].map(
    lambda tid: NBA_TEAM_MAP.get(tid, {}).get("abbrev", "UNK")
)

# Reorder columns to match the required output spec
col_order = [
    "team_id",
    "three_pt_attempt_rate",
    "three_pt_pct",
    "two_pt_pct",
    "ft_point_rate",
    "fga_per_possession",
    "turnover_rate",
    "oreb_rate",
    "transition_rate",
    "off_rating",
    "opp_three_pt_attempt_rate",
    "opp_two_pt_pct",
    "opp_turnover_rate",
    "opp_oreb_rate",
    "def_rating",
    "avg_pace",
    "weighted_off_rapm",
    "star_concentration",
    "rotation_depth",
    "abbrev",
]
df_features = df_features[col_order].reset_index(drop=True)

print(f"Merged shape: {df_features.shape}  (expected: 30 rows x 20 cols)")
df_features.head(5)

## Sanity Checks

In [ ]:
# ---- 1. Basic shape assertion ----
assert len(df_features) == 30, f"Expected 30 teams, got {len(df_features)}"
print("Row count: OK (30 teams)")

# ---- 2. NULL check ----
null_counts = df_features.isnull().sum()
null_cols = null_counts[null_counts > 0]
if null_cols.empty:
    print("NULL check: OK (no nulls)")
else:
    print("WARNING — null values found:")
    print(null_cols)

# ---- 3. Abbrev coverage ----
unk_abbrevs = (df_features["abbrev"] == "UNK").sum()
if unk_abbrevs == 0:
    print("Abbrev mapping: OK (all teams mapped)")
else:
    unmapped = df_features.loc[df_features["abbrev"] == "UNK", "team_id"].tolist()
    print(f"WARNING — {unk_abbrevs} teams not in NBA_TEAM_MAP: {unmapped}")

# ---- 4. Range checks (print warning only, do not assert) ----
RANGE_CHECKS = {
    "three_pt_attempt_rate":      (0.30, 0.55),
    "three_pt_pct":               (0.32, 0.42),
    "two_pt_pct":                 (0.45, 0.65),
    "ft_point_rate":              (0.10, 0.30),
    "fga_per_possession":         (0.55, 0.85),
    "turnover_rate":              (0.08, 0.20),
    "oreb_rate":                  (0.02, 0.12),
    "transition_rate":            (0.00, 0.20),
    "off_rating":                 (95.0, 130.0),
    "opp_three_pt_attempt_rate":  (0.30, 0.55),
    "opp_two_pt_pct":             (0.45, 0.65),
    "opp_turnover_rate":          (0.08, 0.20),
    "opp_oreb_rate":              (0.02, 0.12),
    "def_rating":                 (95.0, 130.0),
    "avg_pace":                   (90.0, 108.0),
    "star_concentration":         (0.00, 1.00),
    "rotation_depth":             (3,    15   ),
}

print("\nRange checks:")
any_warning = False
for col, (lo, hi) in RANGE_CHECKS.items():
    col_min = df_features[col].min()
    col_max = df_features[col].max()
    ok = (col_min >= lo) and (col_max <= hi)
    status = "OK" if ok else "WARNING"
    if not ok:
        any_warning = True
        print(f"  {status:8s} {col:<30s}  actual=[{col_min:.4f}, {col_max:.4f}]  expected=[{lo}, {hi}]")
if not any_warning:
    print("  All features within expected ranges.")

# ---- 5. Display with team abbreviation as index ----
print("\nDescriptive statistics (transposed):")
numeric_cols = [c for c in col_order if c not in ("team_id", "abbrev")]
display_df = df_features.set_index("abbrev")[numeric_cols]
display(display_df.describe().T.round(4))

## Feature Correlation Heatmap

In [ ]:
feature_cols = [c for c in col_order if c not in ("team_id", "abbrev")]
corr_matrix = df_features[feature_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Team Style Feature Correlations", fontsize=14, pad=12)
plt.tight_layout()
os.makedirs("data", exist_ok=True)
plt.savefig("data/feature_correlations.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: data/feature_correlations.png")

## Radar Chart: Top & Bottom Teams per Feature

For a selection of key features, show which teams rank at the top and bottom of the league.

In [ ]:
KEY_FEATURES = [
    ("off_rating",             "Offensive Rating (pts/100 poss)",    True),
    ("def_rating",             "Defensive Rating (pts allowed/100)", False),
    ("three_pt_attempt_rate",  "3PA Rate (3PA / FGA)",               True),
    ("avg_pace",               "Avg Pace (poss/48 min)",             True),
    ("star_concentration",     "Star Concentration (Gini)",          True),
]

df_display = df_features.set_index("abbrev")

for feat, label, higher_is_better in KEY_FEATURES:
    sorted_df = df_display[feat].sort_values(ascending=False)
    direction = "higher = better" if higher_is_better else "lower = better"
    print(f"\n{'='*60}")
    print(f"{label}  [{direction}]")
    print(f"{'='*60}")
    print("Top 3:")
    for abbrev, val in sorted_df.head(3).items():
        print(f"  {abbrev:<4s}  {val:.4f}")
    print("Bottom 3:")
    for abbrev, val in sorted_df.tail(3).items():
        print(f"  {abbrev:<4s}  {val:.4f}")

## Save Features

In [ ]:
os.makedirs("data", exist_ok=True)
df_features.to_csv("data/team_features.csv", index=False)
print(f"Saved {len(df_features)} teams to data/team_features.csv")
print(f"Columns: {list(df_features.columns)}")
conn.close()
print("DB connection closed.")